# 00 — Master Editability: canonical-state & editability synthesis (GRU + refined RSSM)

**The single readable walkthrough of the whole editability / canonical-state investigation**, GRU and
refined RSSM side by side. This is a **consolidation** notebook: cheap artifacts (states, probes, edits,
waterfalls) are recomputed cleanly here; expensive ones (intrinsic dim on the 200k bank, RSSM geometry)
are **cited** from the source notebooks. No `pim` changes — all code lives in the notebook.

Structure (one idea per section, each with a **headline + figure + table**):

- **§0 Premise** — the sim is constant-velocity ⇒ minimal sufficient statistic `(pos,vel)` = 8-dim.
- **§1 Geometry** — low intrinsic dim, curved embedding, fat linear hull.
- **§2 Recoverability** — position (linear/MLP) + the corrected **velocity 2×2** (nonlinear-instantaneous, NOT temporal).
- **§3 Canonicality / fiber** — `h` is non-canonical (~35% not a function of `(pos,vel)`); RSSM det-core **≈ GRU**.
- **§4 Editing head-to-head** — THE CENTREPIECE: five editors, unified waterfall overlay + metrics, incl. the MLP-gradient **reversion**.
- **§5 Synthesis** — predictively-sufficient but non-canonical; readable ≠ controllable; architecture-independent.

Conventions (CLAUDE.md): every code cell tagged `# [N]`; every figure `Fig K — …` with lettered panels
`(a)/(b)/(c)`. Light academic theme for metrics/analysis; **dark** for simulator/observation/waterfall
output. Both rich plots AND printed metric tables in every section. PNGs → `/tmp/master_editability/`.

**Source notebooks (sources of truth):** `canonical_state_editing`, `geodesic_walk_k150`,
`manifold_geometry_diagnostic`, `diagnostic_corrections`, `../rssm_structure/rssm_state_geometry`.
Corrected numbers: `research/scratch/2026-07-08-diagnostic-corrections.md` + `candidate-*.md`.

> **Caveat stated once (applies throughout):** all probes are **in-sample fit** (fit and evaluated on the
> same masked entries, no held-out split). Absolute R²/residual magnitudes are therefore optimistic; the
> **comparisons** (linear-vs-MLP, single-vs-2-frame, GRU-vs-RSSM, det-vs-s, editor-vs-editor) are the
> load-bearing quantities and are unaffected.

---
## Bootstrap — load BOTH models, teacher-force, velocities, shared themes & helpers

In [ ]:
# [1] Shared bootstrap: imports, config, load GRU + refined RSSM, teacher-force, velocities.
import sys, os
sys.path.insert(0, "../../..")   # repo root -> import pim
sys.path.insert(0, "../..")      # notebooks/ -> helpers

from dataclasses import replace
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from IPython.display import display
import h5py

import pim.eval as eval
from pim.extractors import LinearExtractor, MLPExtractor, StateDefinition, identity_mse
from pim.editors import (
    probe_decomposition, inject_state,
    fit_state_subspace, project_to_subspace, offmanifold_residual,
    fit_local_subspace, manifold_steer,
)
from pim.editors.manifold_steering import _pca_subspace
from pim.eval.controllability import _rollout
from pim.world_models import load_checkpoint, load_dataset, make_test_loader

torch.manual_seed(0); np.random.seed(0)

DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE  = 512
NUM_WORKERS = 6
N_OBJ       = 2
DATA_DIR    = "../../../datasets/4_fixed_refl_inview"
OUT = "/tmp/master_editability"; os.makedirs(OUT, exist_ok=True)

GRU_CKPT  = "../../../runs/gru/3_dset3_gru_persistentids_inview_400epochs/best_model.pt"
RSSM_CKPT = "../../../runs/rssm/4_dset4_refined_best/best_model.pt"

# ---- Data (shared) ----
bundle = load_dataset(DATA_DIR, n_obj_keep=N_OBJ)
test, edits = bundle.test, bundle.edits
test_loader = make_test_loader(test, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
DT = float(test.config["dataset"]["sim"]["dt"])

# ---- GRU ----
gru, gru_info = load_checkpoint(GRU_CKPT, device=DEVICE)
H_GRU = gru.hidden_size
preds_gru, states_gru = eval.teacher_force(gru, test_loader, device=DEVICE)   # (N,39,256)

# ---- Refined RSSM (deterministic prior-mean; posterior-mean states) ----
rssm, rssm_info = load_checkpoint(RSSM_CKPT, device=DEVICE)
rssm.sample = False
DET, STO = rssm.cfg.det_size, rssm.cfg.stoch_size
H_RSSM = rssm.hidden_size
preds_rssm, states_rssm = eval.teacher_force(rssm, test_loader, device=DEVICE)  # (N,39,320)
assert DET == 256 and STO == 64 and H_RSSM == 320, (DET, STO, H_RSSM)

print(f"device={DEVICE}  dt={DT}")
print(f"GRU  : {gru_info.run_name} (ep {gru_info.epoch}, val_loss={gru_info.val_loss:.5f})  H={H_GRU}  states={states_gru.shape}")
print(f"RSSM : {rssm_info.run_name} (ep {rssm_info.epoch}, val_loss={rssm_info.val_loss:.5f})  H={H_RSSM} (det={DET},stoch={STO})  states={states_rssm.shape}")

In [ ]:
# [2] Velocities from HDF5 `velocities` (aligned like positions[:, :-1]); flat (pos,vel) targets; visibility.
v_test = h5py.File(test.h5_path, "r")["velocities"][:, :, :N_OBJ, :].astype(np.float32)   # (N,40,2,2)
vel_tf  = v_test[:, :-1, :, :]                        # (N,39,2,2) aligned with states
pos_tf  = test.positions[:, :-1, :N_OBJ, :]          # (N,39,2,2)
vis_tf  = test.is_visible[:, :-1, :N_OBJ].all(axis=2)  # (N,39) both objects visible

print("velocity temporal std (constant-vel sim -> ~0):", float(v_test.std(axis=1).mean()))
print("mean|v| =", float(np.abs(vel_tf).mean()), " (tiny -> depresses absolute vel R², relative story robust)")

posflat_tf = pos_tf.reshape(*pos_tf.shape[:2], N_OBJ*2)               # (N,39,4) [x0,y0,x1,y1]
velflat_tf = vel_tf.reshape(*vel_tf.shape[:2], N_OBJ*2)               # (N,39,4) [vx0,vy0,vx1,vy1]
posvel_tf  = np.concatenate([posflat_tf, velflat_tf], -1)            # (N,39,8)
LATE_T = 15                                                          # late-t = t>=15 (velocity well-determined)

In [ ]:
# [3] Shared THEMES (light for metrics, dark for simulator/waterfall) + generic probe/g-fit helpers.
OK = {"blue":"#0072B2","orange":"#D55E00","green":"#009E73","pink":"#CC79A7","yellow":"#E69F00","grey":"#999999"}
def style_ax(ax):
    ax.spines[["top","right"]].set_visible(False); ax.grid(alpha=0.25, lw=0.6)
plt.style.use("default")

# --- generic MLP/linear regressor: feats -> target, returns (pred, R2_overall, R2_percomp, resid_frac) ---
def _fit_regress(X, Y, kind, hidden=256, n_epochs=100, lr=2e-3, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    Din, Dout = X.shape[1], Y.shape[1]
    Xt = torch.from_numpy(X.astype(np.float32)).to(DEVICE); Yt = torch.from_numpy(Y.astype(np.float32)).to(DEVICE)
    if kind == "linear":
        Xa = torch.cat([Xt, torch.ones(Xt.shape[0],1,device=DEVICE)],1)
        sol = torch.linalg.lstsq(Xa, Yt).solution
        with torch.no_grad(): pred = Xa @ sol
    else:
        net = nn.Sequential(nn.Linear(Din,hidden), nn.ReLU(), nn.Linear(hidden,hidden), nn.ReLU(),
                            nn.Linear(hidden,Dout)).to(DEVICE)
        opt = torch.optim.Adam(net.parameters(), lr=lr); bs = 4096; Nn = Xt.shape[0]
        for ep in range(n_epochs):
            perm = torch.randperm(Nn, device=DEVICE)
            for i in range(0, Nn, bs):
                idx = perm[i:i+bs]
                loss = ((net(Xt[idx]) - Yt[idx])**2).mean()
                opt.zero_grad(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad(): pred = net(Xt)
    resid2 = ((pred - Yt)**2).sum(0)
    tot2   = ((Yt - Yt.mean(0,keepdim=True))**2).sum(0)
    r2_pc  = (1 - resid2/torch.clamp(tot2,min=1e-12)).cpu().numpy()
    r2_all = float(1 - resid2.sum()/tot2.sum())
    resid_frac = float((( (pred-Yt)**2).sum() / (Yt**2).sum()).sqrt())    # ||Y-pred||/||Y||
    return pred.cpu().numpy(), r2_all, r2_pc, resid_frac

def fit_probe(feats_tf, y_tf, mask, kind, **kw):
    """feats_tf:(N,T,F) y_tf:(N,T,D) mask:(N,T)bool. Returns dict on masked entries."""
    X = feats_tf[mask]; Y = y_tf[mask]
    pred, r2, r2pc, rfrac = _fit_regress(X, Y, kind, **kw)
    return dict(r2=r2, r2pc=r2pc, resid_frac=rfrac, n=X.shape[0])

print("themes + helpers ready")

---
## §0 — Premise: the world's minimal sufficient statistic is `(pos, vel)` = 8-dim

**Headline.** The simulator is exactly **constant-velocity** (`pos_{t+1} = pos_t + vel·dt`, dt=1; velocity
never changes). So the *entire* future is determined by the current `(positions, velocities)` — an
**8-dimensional** minimal sufficient statistic for 2 objects. Everything downstream asks whether the
learned hidden state `h` (GRU H=256; RSSM H=320) is a *canonical, factored, editable* carrier of that
8-dim statistic, or merely a predictively-sufficient tangle around it.

In [ ]:
# [4] §0 — verify constant-velocity (temporal std of velocity ~0) and state the DOF budget.
vstd = float(v_test.std(axis=1).mean())
print("="*66); print("§0 PREMISE — constant-velocity sim => (pos,vel) is the sufficient statistic"); print("="*66)
print(f"velocity temporal std over a trajectory : {vstd:.3e}   (~0 => velocity is constant)")
print(f"minimal sufficient statistic            : (pos,vel) = 2 obj x (2 pos + 2 vel) = {N_OBJ*4}-dim")
print(f"learned carriers under test             : GRU h in R^{H_GRU} , RSSM flat in R^{H_RSSM} (det {DET}+stoch {STO})")
print("Question: is h a canonical/factored/editable encoding of that 8-dim statistic, or a tangle around it?")

fig, ax = plt.subplots(figsize=(9,2.2)); ax.axis("off")
boxes = [("world\n(pos,vel)\n8-dim", 0.06, OK["green"]),
         ("GRU h\n256-dim", 0.40, OK["blue"]),
         ("RSSM flat\n320-dim\n(det256+stoch64)", 0.72, OK["orange"])]
for txt,x,c in boxes:
    ax.add_patch(plt.Rectangle((x,0.25),0.18,0.5, fc=c, ec="k", alpha=0.30))
    ax.text(x+0.09,0.5, txt, ha="center", va="center", fontsize=9)
ax.annotate("", xy=(0.40,0.5), xytext=(0.24,0.5), arrowprops=dict(arrowstyle="->",lw=1.6))
ax.annotate("", xy=(0.72,0.5), xytext=(0.24,0.5), arrowprops=dict(arrowstyle="->",lw=1.6,color="0.5"))
ax.text(0.5,0.95,"Fig 0 — the premise: an 8-dim world statistic, learned into fat recurrent states",
        ha="center", fontsize=11)
ax.set_xlim(0,0.92); ax.set_ylim(0,1.05)
fig.savefig(f"{OUT}/fig0_premise.png", dpi=130, bbox_inches="tight"); display(fig); plt.close(fig)

---
## §1 — Geometry: a low-intrinsic-dim, strongly curved surface in a fat linear hull

**Headline.** The visited-state manifold has **intrinsic dimension ≈ 5–7** (model-free TwoNN 5.2 / MLE 6.9),
bracketing the physical **8 DOF** — but it lives in a much **fatter linear hull** (GRU 38 dims @90% var) because
it is **strongly curved**: local tangent planes reorient ~56° (GRU) / ~65° (RSSM) at nearest-neighbour spacing.
The RSSM replicates this, slightly more curved (34/320 @90%, 65°). *Curvature is the geometric reason linear /
min-norm probe edits leave the manifold* (→ §4).

*Recompute here (cheap):* PCA scree for both models. *Cite (expensive, 200k-bank):* intrinsic dim & tangent
angles from `manifold_geometry_diagnostic` (GRU) and `rssm_state_geometry` (RSSM).

In [ ]:
# [5] §1 — recompute PCA scree for BOTH models (cheap); cite intrinsic-dim & curvature (expensive bank).
def scree(states, H):
    bank = states.reshape(-1, H)
    sub = _pca_subspace(torch.from_numpy(bank).float().to(DEVICE), n_components=H, var_threshold=1.0)
    ratio = sub.explained_variance_ratio.cpu().numpy(); cum = np.cumsum(ratio)
    dims = {p:int((cum<p).sum())+1 for p in (0.70,0.90,0.95)}
    return ratio, cum, dims

ratio_g, cum_g, dims_g = scree(states_gru, H_GRU)
ratio_r, cum_r, dims_r = scree(states_rssm, H_RSSM)

CITED = dict(  # from source notebooks (expensive 200k-bank / RSSM geometry)
    intrinsic_twonn=5.2, intrinsic_mle=6.9, physical_dof=8,
    tangent_angle_gru=56.0, tangent_angle_rssm=65.2,
    dims90_gru_cited=38, dims90_rssm_cited=34,
    honest_local_resid_real="0.75-0.84", global_flat_resid_real=1.75)

print("=== §1 GEOMETRY TABLE (recomputed scree + cited intrinsic dim / curvature) ===")
print(f"{'quantity':34s} {'GRU':>10s} {'RSSM':>10s}")
print(f"{'PCA dims @70% var (recomputed)':34s} {dims_g[0.70]:>10d} {dims_r[0.70]:>10d}")
print(f"{'PCA dims @90% var (recomputed)':34s} {dims_g[0.90]:>10d} {dims_r[0.90]:>10d}")
print(f"{'PCA dims @95% var (recomputed)':34s} {dims_g[0.95]:>10d} {dims_r[0.95]:>10d}")
print(f"{'  (cited @90% from sources)':34s} {CITED['dims90_gru_cited']:>10d} {CITED['dims90_rssm_cited']:>10d}")
print(f"{'intrinsic dim TwoNN (cited)':34s} {CITED['intrinsic_twonn']:>10} {'~same':>10}")
print(f"{'intrinsic dim MLE (cited)':34s} {CITED['intrinsic_mle']:>10} {'~same':>10}")
print(f"{'physical DOF':34s} {CITED['physical_dof']:>10} {CITED['physical_dof']:>10}")
print(f"{'tangent rotation @NN spacing (cited)':34s} {str(CITED['tangent_angle_gru'])+'deg':>10} {str(CITED['tangent_angle_rssm'])+'deg':>10}")
print("\nRead: intrinsic ~5-7 (brackets 8 DOF) << linear-hull ~34-38 @90% => curved embedding.")

In [ ]:
# [6] Fig 1 — geometry: (a) scree GRU vs RSSM, (b) intrinsic vs hull dims, (c) tangent-rotation curvature bar.
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
ax = axes[0]
ax.plot(np.arange(1,len(cum_g)+1), cum_g, color=OK["blue"], lw=2, label="GRU")
ax.plot(np.arange(1,len(cum_r)+1), cum_r, color=OK["orange"], lw=2, label="RSSM")
ax.axhline(0.90,color="0.6",ls=":",lw=1)
ax.axvline(dims_g[0.90], color=OK["blue"], ls="--", lw=1); ax.axvline(dims_r[0.90], color=OK["orange"], ls="--", lw=1)
ax.set_xlim(0,80); ax.set_xlabel("# PCA components"); ax.set_ylabel("cumulative variance")
ax.set_title(f"(a) PCA scree: hull {dims_g[0.90]}/{H_GRU} (GRU) vs {dims_r[0.90]}/{H_RSSM} (RSSM) @90%"); ax.legend(fontsize=8); style_ax(ax)
ax = axes[1]
labels = ["TwoNN\nintrinsic","MLE\nintrinsic","physical\nDOF","hull@90%\nGRU","hull@90%\nRSSM"]
vals   = [CITED["intrinsic_twonn"], CITED["intrinsic_mle"], CITED["physical_dof"], dims_g[0.90], dims_r[0.90]]
cols   = [OK["green"],OK["green"],"k",OK["blue"],OK["orange"]]
ax.bar(labels, vals, color=cols, alpha=0.85)
for i,v in enumerate(vals): ax.text(i, v+0.6, f"{v:g}", ha="center", fontsize=8)
ax.set_ylabel("dimension"); ax.set_title("(b) intrinsic ~5-7 << linear hull => curvature"); style_ax(ax)
ax = axes[2]
ax.bar(["GRU","RSSM"], [CITED["tangent_angle_gru"],CITED["tangent_angle_rssm"]],
       color=[OK["blue"],OK["orange"]], alpha=0.85)
for i,v in enumerate([CITED["tangent_angle_gru"],CITED["tangent_angle_rssm"]]): ax.text(i,v+1,f"{v:.0f}deg",ha="center")
ax.axhline(30, color="0.6", ls=":", lw=1, label="30deg curvature scale")
ax.set_ylabel("tangent rotation @ NN spacing"); ax.set_ylim(0,80)
ax.set_title("(c) local tangent reorients strongly (cited)"); ax.legend(fontsize=8); style_ax(ax)
fig.suptitle("Fig 1 — State geometry: low intrinsic dim, strongly curved, fat linear hull (GRU + RSSM)", y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig1_geometry.png", dpi=130, bbox_inches="tight"); display(fig); plt.close(fig)

---
## §2 — Recoverability: position is (nearly) linear; **velocity is nonlinear-instantaneous, NOT temporal**

**Headline.** Position is well read out of a single `h_t` (linear R² ≈ 0.84, MLP ≈ 0.96). **Velocity is
instantaneously readable from a single frame — but only NONLINEARLY** (single-frame linear ≈ 0.5–0.59 →
single-frame **MLP ≈ 0.94 late-t**); a 2-frame window adds essentially nothing (Δ ≤ 0.015 all-t, ≤ 0.007
late-t on both models), and differencing `dh = h_t − h_{t-1}` is strictly *worse*. **The old "velocity is
temporal (0.47 linear → 0.76 2-frame MLP)" reading was a linear-vs-MLP confound and is RETIRED.**

For the RSSM we also split **h-only / s-only / full**: position lives in the **deterministic `h`**, not the
stochastic `s` (det-only pos R² ≈ full; s-only much lower).

In [ ]:
# [7] §2 — velocity 2x2 {linear,MLP}x{single-frame,2-frame}, both models, all-t & late-t; + dh.
VCOMP = ["vx0","vy0","vx1","vy1"]
def build_feats(states, late=False):
    sf  = states[:, 1:, :]                                            # h_t
    win = np.concatenate([states[:, :-1, :], states[:, 1:, :]], -1)   # [h_{t-1},h_t]
    dh  = states[:, 1:, :] - states[:, :-1, :]
    y   = velflat_tf[:, 1:, :]
    mask = vis_tf[:, 1:] & vis_tf[:, :-1]
    if late:
        lm = np.zeros_like(mask); lm[:, LATE_T-1:] = True; mask = mask & lm
    return sf, win, dh, y, mask

def run_vel_2x2(states, late=False):
    sf, win, dh, y, mask = build_feats(states, late)
    o = {}
    o[("sf","lin")]  = fit_probe(sf, y, mask, "linear")
    o[("sf","mlp")]  = fit_probe(sf, y, mask, "mlp")
    o[("win","lin")] = fit_probe(win, y, mask, "linear")
    o[("win","mlp")] = fit_probe(win, y, mask, "mlp")
    o[("dh","mlp")]  = fit_probe(dh, y, mask, "mlp")
    return o

vel_res = {}
for lbl, st in [("GRU",states_gru),("RSSM",states_rssm)]:
    for reg,late in [("all",False),("late",True)]:
        vel_res[(lbl,reg)] = run_vel_2x2(st, late)

print("=== VELOCITY 2x2 (overall R2) — single-frame MLP ~= 2-frame MLP => nonlinear-INSTANTANEOUS ===")
print(f"{'model/reg':12s} {'sf-lin':>7s} {'sf-MLP':>7s} {'2f-lin':>7s} {'2f-MLP':>7s} {'dh-MLP':>7s} {'D(2f-sf MLP)':>13s}")
for lbl in ["GRU","RSSM"]:
    for reg in ["all","late"]:
        o = vel_res[(lbl,reg)]; d = o[("win","mlp")]["r2"]-o[("sf","mlp")]["r2"]
        print(f"{lbl+'/'+reg:12s} {o[('sf','lin')]['r2']:7.3f} {o[('sf','mlp')]['r2']:7.3f} "
              f"{o[('win','lin')]['r2']:7.3f} {o[('win','mlp')]['r2']:7.3f} {o[('dh','mlp')]['r2']:7.3f} {d:+13.4f}")
print("\nAll D(2-frame - single-frame MLP) <= 0.015 => temporal window adds ~nothing => 'velocity temporal' RETIRED.")

In [ ]:
# [8] §2 — position recoverability (linear vs MLP) + RSSM h-only/s-only/full position split.
pos_res = {}
for lbl, st in [("GRU",states_gru),("RSSM",states_rssm)]:
    pos_res[(lbl,"lin")] = fit_probe(st, posflat_tf, vis_tf, "linear")
    pos_res[(lbl,"mlp")] = fit_probe(st, posflat_tf, vis_tf, "mlp")

h_det   = states_rssm[..., :DET]; s_stoch = states_rssm[..., DET:]
pos_split = {
    "RSSM full (320)": fit_probe(states_rssm, posflat_tf, vis_tf, "linear")["r2"],
    "RSSM h_det (256)": fit_probe(h_det,      posflat_tf, vis_tf, "linear")["r2"],
    "RSSM s_stoch (64)": fit_probe(s_stoch,   posflat_tf, vis_tf, "linear")["r2"],
}
print("=== POSITION recoverability (overall R2) ===")
print(f"{'model':12s} {'linear':>8s} {'MLP':>8s}")
for lbl in ["GRU","RSSM"]:
    print(f"{lbl:12s} {pos_res[(lbl,'lin')]['r2']:8.3f} {pos_res[(lbl,'mlp')]['r2']:8.3f}")
print("\n=== RSSM position split (linear): does position live in det h or stochastic s? ===")
for k,v in pos_split.items(): print(f"  {k:20s} R2={v:.3f}")
print("  -> position lives in the DETERMINISTIC h (det~=full >> s-only).")

In [ ]:
# [9] Fig 2 — recoverability: (a) velocity 2x2 bars per model, (b) per-comp vel R2 late-t, (c) position + RSSM split.
fig, axes = plt.subplots(1, 3, figsize=(17, 4.4))
ax = axes[0]
setk = [("sf","lin"),("sf","mlp"),("win","lin"),("win","mlp"),("dh","mlp")]
xl = ["lin\nh_t","MLP\nh_t","lin\n[h-1,h]","MLP\n[h-1,h]","MLP\ndh"]
x = np.arange(len(setk)); w=0.2
for lbl,shift,c in [("GRU",-1.5,OK["blue"]),("RSSM",0.5,OK["orange"])]:
    for reg,alpha,off in [("all",0.5,0),("late",1.0,1)]:
        vals = [vel_res[(lbl,reg)][k]["r2"] for k in setk]
        ax.bar(x + (shift+off)*w, vals, w, color=c, alpha=alpha, label=f"{lbl} {reg}-t")
ax.set_xticks(x); ax.set_xticklabels(xl, fontsize=8); ax.set_ylabel("velocity R2 (overall)")
ax.axhline(1,color="0.6",ls=":",lw=1); ax.set_title("(a) velocity 2x2: linear<<MLP, single~=2-frame"); ax.legend(fontsize=6); style_ax(ax)
ax = axes[1]
x = np.arange(4); w=0.2
for j,(lbl,c) in enumerate([("GRU",OK["blue"]),("RSSM",OK["orange"])]):
    for feat,alpha,off in [("sf",1.0,0),("win",0.5,1)]:
        pc = vel_res[(lbl,"late")][(feat,"mlp")]["r2pc"]
        ax.bar(x + (j*2+off-1.5)*w, pc, w, color=c, alpha=alpha, label=f"{lbl} {'1f' if feat=='sf' else '2f'}")
ax.set_xticks(x); ax.set_xticklabels(VCOMP); ax.set_ylabel("velocity R2 per comp (late-t, MLP)")
ax.set_ylim(0,1.02); ax.set_title("(b) velocity cleanly snapshot-readable (all >=0.88)"); ax.legend(fontsize=6); style_ax(ax)
ax = axes[2]
x = np.arange(2); w=0.35
ax.bar(x-w/2, [pos_res[("GRU","lin")]["r2"],pos_res[("RSSM","lin")]["r2"]], w, color=OK["blue"], label="linear")
ax.bar(x+w/2, [pos_res[("GRU","mlp")]["r2"],pos_res[("RSSM","mlp")]["r2"]], w, color=OK["orange"], label="MLP")
ax.set_xticks(x); ax.set_xticklabels(["GRU","RSSM"]); ax.set_ylabel("position R2"); ax.set_ylim(0,1.02)
txt = "RSSM pos (linear):\n full {full:.2f} | det {det:.2f} | s {s:.2f}".format(
    full=pos_split["RSSM full (320)"], det=pos_split["RSSM h_det (256)"], s=pos_split["RSSM s_stoch (64)"])
ax.text(0.98,0.05, txt, transform=ax.transAxes, ha="right", va="bottom", fontsize=7, bbox=dict(fc="0.95",ec="0.7"))
ax.set_title("(c) position: nearly linear; lives in det h"); ax.legend(fontsize=8); style_ax(ax)
fig.suptitle("Fig 2 — Recoverability: position ~linear; velocity nonlinear-instantaneous (NOT temporal)", y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig2_recoverability.png", dpi=130, bbox_inches="tight"); display(fig); plt.close(fig)

---
## §3 — Canonicality / fiber-collapse: `h` is non-canonical, and the RSSM det-core is **no more canonical**

**Headline.** Fit the best nonlinear `g(pos,vel) → block` and measure the residual fraction
`‖block − g‖ / ‖block‖` (lower ⇒ more nearly a *function of* the sufficient statistic ⇒ more canonical). The
GRU `h` leaves **residual ≈ 0.337** — i.e. **~34% of `h` is NOT a function of `(pos,vel)`** (it carries
history / scaffolding beyond the minimal statistic). The strong linear→MLP drop confirms a **curved** embedding.

**Correction (2026-07-08):** the RSSM's **deterministic core** leaves **≈ 0.368 ≈ GRU 0.337** — essentially
on par (do NOT read the ~0.03 gap as a real architectural difference). The full-320 number (≈0.602) was
**inflated by the stochastic `s`** (its own residual ≈0.891 — legitimately not a function of `(pos,vel)`, as
expected for a KL-regularised latent). **The KL structure buys no canonicity.**

In [ ]:
# [10] §3 — fiber-collapse residual: fit g(pos,vel)->block (linear & MLP) for GRU h, RSSM full/det/s.
def fit_g(block, kind):
    X = posvel_tf[vis_tf]; Y = block[vis_tf]
    _, r2, _, rfrac = _fit_regress(X, Y, kind, hidden=512, n_epochs=120, lr=1.5e-3)
    return rfrac, r2

blocks = {"GRU h (256)":states_gru, "RSSM full (320)":states_rssm,
          "RSSM h_det (256)":h_det, "RSSM s_stoch (64)":s_stoch}
fiber = {}
print("=== FIBER-COLLAPSE: residual fraction ||block - g(pos,vel)|| / ||block||  (lower = more canonical) ===")
print(f"{'block':22s} {'lin resid':>10s} {'lin R2':>8s} {'MLP resid':>10s} {'MLP R2':>8s}")
for name, blk in blocks.items():
    lrf,lr2 = fit_g(blk,"linear"); mrf,mr2 = fit_g(blk,"mlp")
    fiber[name] = dict(lin=(lrf,lr2), mlp=(mrf,mr2))
    print(f"{name:22s} {lrf:10.4f} {lr2:8.4f} {mrf:10.4f} {mr2:8.4f}")

gru_r=fiber["GRU h (256)"]["mlp"][0]; det_r=fiber["RSSM h_det (256)"]["mlp"][0]
full_r=fiber["RSSM full (320)"]["mlp"][0]; s_r=fiber["RSSM s_stoch (64)"]["mlp"][0]
print(f"\nHEADLINE: GRU h={gru_r:.3f}  ~=  RSSM det-only={det_r:.3f}   (~0.03 gap - do NOT over-read)")
print(f"          stochastic s inflates full: full {full_r:.3f} vs det {det_r:.3f} (D={full_r-det_r:+.3f}); s-block itself {s_r:.3f}")
print(f"          ~{gru_r*100:.0f}% of GRU h is NOT a function of (pos,vel) => non-canonical; KL structure adds no canonicity.")

In [ ]:
# [11] Fig 3 — fiber residual bars: (a) residual fraction linear vs MLP, (b) R2 on block.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
order = ["GRU h (256)","RSSM full (320)","RSSM h_det (256)","RSSM s_stoch (64)"]
x = np.arange(len(order)); w=0.38
ax = axes[0]
lin_rf=[fiber[n]["lin"][0] for n in order]; mlp_rf=[fiber[n]["mlp"][0] for n in order]
ax.bar(x-w/2, lin_rf, w, label="linear g", color=OK["blue"])
ax.bar(x+w/2, mlp_rf, w, label="MLP g", color=OK["orange"])
for xi,v in zip(x+w/2, mlp_rf): ax.text(xi, v+0.01, f"{v:.3f}", ha="center", fontsize=8)
ax.axhline(gru_r, color=OK["green"], ls=":", lw=1.2, label=f"GRU MLP resid={gru_r:.3f}")
ax.set_xticks(x); ax.set_xticklabels(order, rotation=18, ha="right", fontsize=8)
ax.set_ylabel("residual fraction ||blk-g||/||blk||"); ax.set_title("(a) is the block a function of (pos,vel)?")
ax.legend(fontsize=7); style_ax(ax)
ax = axes[1]
lin_r2=[fiber[n]["lin"][1] for n in order]; mlp_r2=[fiber[n]["mlp"][1] for n in order]
ax.bar(x-w/2, lin_r2, w, label="linear g", color=OK["blue"])
ax.bar(x+w/2, mlp_r2, w, label="MLP g", color=OK["orange"])
ax.set_xticks(x); ax.set_xticklabels(order, rotation=18, ha="right", fontsize=8)
ax.set_ylabel("R2 on block"); ax.set_ylim(0,1.0); ax.set_title("(b) variance of block explained by g(pos,vel)")
ax.legend(fontsize=7); style_ax(ax)
fig.suptitle("Fig 3 — Fiber collapse: GRU h non-canonical (~34%); RSSM det-core ~= GRU; stochastic s holds none", y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig3_fiber.png", dpi=130, bbox_inches="tight"); display(fig); plt.close(fig)

---
## §4 — Editing head-to-head (THE CENTREPIECE): readable ≠ controllable

We compare **five** editors on the **same** edits (warm up to `edit_frame=20`, roll out ~15 steps), GRU
primary. Target = post-edit teleported position at the edit frame.

| editor | what it is | one line |
|---|---|---|
| **GT** | true post-edit teacher-forced rollout | the observation a perfect edit should produce |
| **Unsteered** | no edit (cold-start rollout) | baseline: object stays at the pre-edit location (ghost) |
| **Manifold-global** | one-shot POCS: inject readout ↔ project onto **global PCA** (`manifold_steer`) | on-manifold but reaches readout only partially |
| **PCA geodesic** | **iterative** constant-step local-tangent walk toward the readout (refit fresh local tangent each step) | *not* a one-shot local projection — an iterative walk |
| **MLP-gradient** | **obs-driven** Adam on `h` to match the GT observation | hits the observation at step 0, then **reverts** |

**Naming (kept distinct):** "local-tangent projection" = one-shot; **"PCA geodesic"** = the iterative walk.

**Two deliverables:** (a) a big, clear **unified waterfall overlay** (green = target loc, red = ghost) with a
column per editor and the GT/observation waterfalls (dark theme); (b) a **metrics table** (→target render,
obs-change, ghost ratio, per-step persistence, off-manifold residual). Plus the **reversion example**: the
MLP-gradient edit reaches the target at **step 0** and **reverts by ~step 4** — shown as per-step →target /
ghost curves AND the waterfall snapping back.

> *Metric caveat (stated once):* the "obs-change % of a full swap" uses a **weak pseudoinverse denominator**
> (`diagnostic_corrections` §3) — read it as "moves a lot," not a clean 0–100%. We also print a **teacher-forced
> true-post-edit (GT) reference** as a proper 100% baseline where available.

In [ ]:
# [12] §4 — warm up to edit frame; build position probe, targets, subspace + local bank (GRU).
from tqdm.auto import tqdm
model = gru; H = H_GRU; states_edit = states_gru
SUBSPACE_VAR, LOCAL_VAR, LOCAL_BANK_SIZE = 0.90, 0.90, 50_000
LOCAL_K_GEO = 64                    # honest small-k (diagnostic_corrections used {16,32,64}; 64 = best reach)
N_EDIT, N_ROLLOUT = 64, 15
K_GEO_ITERS = 120

# linear position probe on teacher-forced states
sdef = StateDefinition(name="positions", state_shape=(N_OBJ,2), extract_fn=lambda b: b["positions"])
linear_pos = LinearExtractor(H, sdef, use_lstsq=True)
linear_pos.fit(states_edit, pos_tf, mask=vis_tf, device=DEVICE)
linear_pos = linear_pos.to(DEVICE).eval()
A, b_, A_pinv = probe_decomposition(linear_pos)

# global subspace + local bank
subspace = fit_state_subspace(states_edit, var_threshold=SUBSPACE_VAR)
subspace_dev = replace(subspace, mean=subspace.mean.to(DEVICE), basis=subspace.basis.to(DEVICE),
                       explained_variance_ratio=subspace.explained_variance_ratio.to(DEVICE))
_bank_all = states_edit.reshape(-1, H)
_sub = np.random.RandomState(0).choice(_bank_all.shape[0], size=min(LOCAL_BANK_SIZE,_bank_all.shape[0]), replace=False)
bank_dev = torch.from_numpy(_bank_all[_sub]).float().to(DEVICE)

# warm up + targets
N = min(N_EDIT, edits.n_samples)
warm = eval.warm_up_to_edit(model, edits.obs[:N], edits.edit_frame, n_viz=N, n_ctx_show=8, device=DEVICE)
h0 = torch.from_numpy(warm.h_at_edit[:N]).float().to(DEVICE)
ef = edits.edit_frame
targets = edits.positions[:N, ef, :N_OBJ, :].reshape(N, N_OBJ*2)
tgt = torch.from_numpy(targets).float().to(DEVICE)

def readout(h): return h @ A.T + b_
def readout_rmse(h): return float((readout(h)-tgt).pow(2).mean().sqrt())
def resid_global(h): return float(offmanifold_residual(h, subspace_dev).mean())
print(f"N={N}  edit_frame={ef}  cold-start readout RMSE={readout_rmse(h0):.4f}")

In [ ]:
# [13] §4 — honest leave-out local off-manifold residual (curvature-aware; NOT the projection tautology).
@torch.no_grad()
def honest_local_resid(h_batch, k_neighbors=LOCAL_K_GEO, leave_out=True, n_probe=100, var_threshold=LOCAL_VAR):
    hb = h_batch if isinstance(h_batch, torch.Tensor) else torch.as_tensor(h_batch, device=DEVICE, dtype=torch.float32)
    fracs = []
    npb = min(n_probe, hb.shape[0])
    for i in range(npb):
        q = hb[i].reshape(-1)
        d = torch.cdist(q[None], bank_dev)[0]
        kk = k_neighbors + (1 if leave_out else 0)
        idx = torch.topk(d, min(kk, bank_dev.shape[0]), largest=False).indices
        if leave_out: idx = idx[1:]
        sub = _pca_subspace(bank_dev[idx], n_components=None, var_threshold=var_threshold)
        proj = project_to_subspace(q[None], sub)[0]
        raw = float((q - proj).norm()); denom = float((q - sub.mean).norm())
        fracs.append(raw / max(denom,1e-9))
    return float(np.mean(fracs))

real_states_probe = torch.from_numpy(_bank_all[_sub[:200]]).float().to(DEVICE)
real_honest = honest_local_resid(real_states_probe, n_probe=200)
print(f"honest leave-out local residual of REAL states (k={LOCAL_K_GEO}) = {real_honest:.4f}  (the on-manifold reference)")

In [ ]:
# [14] §4 — the three latent editors: Manifold-global (POCS), PCA-geodesic (iterative), MLP-gradient (obs-driven).
edit_fn = lambda h, t: inject_state(h, t, A, A_pinv, b_)

# (1) Manifold-global: one-shot inject<->project-to-global-PCA (POCS)
h_manifold = manifold_steer(h0, tgt, edit_fn, subspace_dev, n_iters=50)

# (2) PCA geodesic: ITERATIVE constant-step local-tangent walk (fresh local tangent each iter)
with torch.no_grad():
    d0 = (inject_state(h0, tgt, A, A_pinv, b_) - h0).norm(dim=-1)
CONST_STEP = 0.34 * float(d0.mean())     # matches geodesic_walk_k150 / diagnostic_corrections step size
@torch.no_grad()
def pca_geodesic(h_start, target, k_local=LOCAL_K_GEO, const_step=CONST_STEP, k_iters=K_GEO_ITERS):
    Nn = h_start.shape[0]; h_out = torch.empty_like(h_start)
    rmse_log = np.zeros((Nn, k_iters+1))
    for i in tqdm(range(Nn), desc="PCA geodesic", leave=False):
        h = h_start[i:i+1]; t = target[i:i+1]
        rmse_log[i,0] = float((readout(h)-t).pow(2).mean().sqrt())
        for kk in range(k_iters):
            d = inject_state(h, t, A, A_pinv, b_) - h; nrm = d.norm()
            dhat = d/nrm if float(nrm)>1e-12 else d
            h_step = h + const_step*dhat
            sub = fit_local_subspace(bank_dev, h_step[0], k_neighbors=k_local, var_threshold=LOCAL_VAR, bank_size=LOCAL_BANK_SIZE)
            h = project_to_subspace(h_step, sub)
            rmse_log[i,kk+1] = float((readout(h)-t).pow(2).mean().sqrt())
        h_out[i] = h[0]
    return h_out, rmse_log
h_geodesic, geo_rmse_log = pca_geodesic(h0, tgt)

# (3) MLP-gradient: obs-driven Adam on h to match the GT post-edit observation at the edit frame.
# (cudnn RNN backward needs train mode; disable cudnn so backward runs on the eval GRU - identical forward.)
gt_obs_single = torch.from_numpy(edits.clean_obs[:N, ef, :]).float().to(DEVICE)   # GT obs a perfect edit produces
def obs_grad_edit(h_init, target_obs, n_iter=400, lr=0.05):
    h = h_init.clone().detach().requires_grad_(True); opt = torch.optim.Adam([h], lr=lr)
    with torch.backends.cudnn.flags(enabled=False):
        for _ in range(n_iter):
            pred = model.decode(model.state_from_flat(h))
            loss = ((pred - target_obs)**2).mean()
            opt.zero_grad(); loss.backward(); opt.step()
    return h.detach(), float(loss.item())
h_mlpgrad, mlp_final_loss = obs_grad_edit(h0, gt_obs_single)

print(f"CONST_STEP={CONST_STEP:.4f}   geodesic readout RMSE {geo_rmse_log[:,0].mean():.3f}->{geo_rmse_log[:,-1].mean():.3f}")
print(f"MLP-gradient final decode loss vs GT obs = {mlp_final_loss:.6f}")
print("readout RMSE:  unsteered {:.3f} | manifold {:.3f} | geodesic {:.3f} | MLP-grad {:.3f}".format(
    readout_rmse(h0), readout_rmse(h_manifold), readout_rmse(h_geodesic), readout_rmse(h_mlpgrad)))

In [ ]:
# [15] §4 — GT reference state (teacher-force the TRUE post-edit trajectory to edit_frame) + rollouts of all editors.
@torch.no_grad()
def tf_hidden_at(obs_seqs, frame):
    out = np.zeros((obs_seqs.shape[0], H), np.float32)
    for i in range(obs_seqs.shape[0]):
        ot = torch.from_numpy(obs_seqs[i]).float().to(DEVICE); state=None
        for t in range(frame+1):
            _, state = model.step(ot[t].unsqueeze(0), state)
        out[i] = model.flat_state(state).squeeze(0).cpu().numpy()
    return out
h_gt = torch.from_numpy(tf_hidden_at(edits.obs[:N], ef)).float().to(DEVICE)   # canonical/GT post-edit state

@torch.no_grad()
def rollout_from_flat(h_array, n_rollout):
    obs_all=[]
    for i in range(h_array.shape[0]):
        h = torch.as_tensor(h_array[i], dtype=torch.float32, device=DEVICE).unsqueeze(0)
        o,_ = _rollout(model, h, n_rollout); obs_all.append(o)
    return np.stack(obs_all)

EDITORS = {"GT": h_gt, "Unsteered": h0, "Manifold-global": h_manifold,
           "PCA geodesic": h_geodesic, "MLP-gradient": h_mlpgrad}
roll_obs = {n: rollout_from_flat(h.detach().cpu().numpy(), N_ROLLOUT) for n,h in EDITORS.items()}
OBS_RES = roll_obs["Unsteered"].shape[-1]
print("rollouts:", {k:v.shape for k,v in roll_obs.items()})

In [ ]:
# [16] §4 — build TARGET / PRE renders (for obs-space metrics + waterfall centroids) and ghost mask.
from pim.simulator.sim import Scene, SimConfig
from pim.simulator.renderer import render_scene
sim = test.config["dataset"]["sim"]
def make_cfg(nf):
    return SimConfig(seed=0, y_near=sim["y_near"], y_far=sim["y_far"], x_near=sim["x_near"], x_far=sim["x_far"],
                     n_objects=N_OBJ, radius=sim["radius"], n_frames=nf, dt=sim["dt"], obs_res=sim["obs_res"],
                     refl_min=sim["refl_min"], refl_max=sim["refl_max"], fixed_reflectivities=True,
                     obs_noise_std=0.0, boundary="open", always_in_frustum=False)
REFL = np.array([sim["refl_min"], sim["refl_max"]], dtype=np.float32)
RAD  = np.array([sim["radius"]]*N_OBJ, dtype=np.float32)
COLc = np.tile(np.array([[1,1,1]], np.float32), (N_OBJ,1))
tgt_pos = edits.positions[:N, ef, :N_OBJ, :].astype(np.float32)
pre_pos = edits.positions[:N, ef-1, :N_OBJ, :].astype(np.float32)
cfg1 = make_cfg(1)
tgt_render_id  = np.zeros((N,OBS_RES), np.int64); tgt_render_int = np.zeros((N,OBS_RES), np.float32)
pre_render_id  = np.zeros((N,OBS_RES), np.int64)
for i in range(N):
    sc = Scene(positions=tgt_pos[i][None], velocities=np.zeros((1,N_OBJ,2),np.float32),
               radii=RAD, colors=COLc, reflectivities=REFL, config=cfg1)
    _, rid, rint = render_scene(sc); tgt_render_id[i],tgt_render_int[i]=rid[0],rint[0]
    scp = Scene(positions=pre_pos[i][None], velocities=np.zeros((1,N_OBJ,2),np.float32),
                radii=RAD, colors=COLc, reflectivities=REFL, config=cfg1)
    _, ridp, _ = render_scene(scp); pre_render_id[i]=ridp[0]
edit_obj = edits.edit_object[:N]
ghost_mask = np.zeros((N,OBS_RES), bool)
for i in range(N):
    ghost_mask[i] = (pre_render_id[i]==edit_obj[i]) & (tgt_render_id[i]!=edit_obj[i])
print("renders built; ghost rays available:", int(ghost_mask.sum()))

In [ ]:
# [17] §4 — obs-space metrics: ->target, obs-change (% of GT swap), ghost, per-step persistence, off-manifold resid.
obs_u = roll_obs["Unsteered"]
def rms(a,b): return float(np.sqrt(((a-b)**2).mean()))
def dist_to_target(obs, s=0): return rms(obs[:,s,:], tgt_render_int)
def obs_change(obs, s=0):     return rms(obs[:,s,:], obs_u[:,s,:])
def ghost_ratio(obs, s=0):
    if ghost_mask.sum()==0: return np.nan
    return float(obs[:,s,:][ghost_mask].mean() / max(obs_u[:,s,:][ghost_mask].mean(),1e-6))

# 100% swap references: GT (proper true-post-edit) AND pseudoinverse (weak denominator, for context).
h_pinv = inject_state(h0, tgt, A, A_pinv, b_)
gt_swap  = rms(roll_obs["GT"][:,0,:],  obs_u[:,0,:])                    # proper 100% reference
pinv_swap = rms(rollout_from_flat(h_pinv.detach().cpu().numpy(), 1)[:,0,:], obs_u[:,0,:])  # weak denom

# per-editor honest off-manifold residual (leave-out local) + global residual
honest_ed = {n: honest_local_resid(h, n_probe=min(64,N)) for n,h in EDITORS.items() if n!="GT"}
honest_ed["GT"] = honest_local_resid(h_gt, n_probe=min(64,N))

print(f"GT swap obs-change (proper 100% ref) = {gt_swap:.4f}   |   pseudoinv swap (weak denom) = {pinv_swap:.4f}")
print(f"real-state honest off-manifold residual reference = {real_honest:.3f}\n")
print("=== §4 EDITOR METRICS (rollout step 0 = direct edit) ===")
print(f"{'editor':16s} {'readoutRMSE':>11s} {'->target':>9s} {'obschg':>8s} {'%GTswap':>8s} {'ghost':>7s} {'honestResid':>12s} {'globResid':>10s}")
metrics4 = {}
for n,h in EDITORS.items():
    obs = roll_obs[n]; rr = readout_rmse(h) if n!="GT" else float((readout(h_gt)-tgt).pow(2).mean().sqrt())
    dt_,dc,g = dist_to_target(obs), obs_change(obs), ghost_ratio(obs)
    pct = 100*dc/max(gt_swap,1e-9)
    metrics4[n]=dict(readout=rr, to_target=dt_, obs_chg=dc, pct=pct, ghost=g, honest=honest_ed[n], glob=resid_global(h))
    print(f"{n:16s} {rr:11.4f} {dt_:9.4f} {dc:8.4f} {pct:8.1f} {g:7.3f} {honest_ed[n]:12.3f} {resid_global(h):10.3f}")
print(f"\nreadable != controllable: probe/geodesic hit the READOUT but the OBS barely moves toward target;")
print(f"MLP-gradient hits the OBS at step0 but is OFF-manifold (honest resid {honest_ed['MLP-gradient']:.2f} vs real {real_honest:.2f}) and reverts (next cell).")

In [ ]:
# [18] §4 — THE REVERSION: per-step ->target and ghost for MLP-gradient (reaches at step0, reverts by ~step4).
steps = np.arange(N_ROLLOUT)
to_tgt_step = {n:[dist_to_target(o,s) for s in steps] for n,o in roll_obs.items()}
ghost_step  = {n:[ghost_ratio(o,s)    for s in steps] for n,o in roll_obs.items()}
print("=== MLP-GRADIENT REVERSION (per rollout step) ===")
print(f"{'step':>4s} {'->target':>9s} {'ghost':>7s}   (GT ->target for scale)")
for s in steps:
    print(f"{s:>4d} {to_tgt_step['MLP-gradient'][s]:9.4f} {ghost_step['MLP-gradient'][s]:7.3f}   {to_tgt_step['GT'][s]:9.4f}")
s0 = to_tgt_step["MLP-gradient"][0]; s4 = to_tgt_step["MLP-gradient"][min(4,N_ROLLOUT-1)]
print(f"\n->target: step0={s0:.3f} (near GT {to_tgt_step['GT'][0]:.3f}) -> step4={s4:.3f}  => REVERTS ({100*(s4-s0)/max(s0,1e-9):+.0f}% back toward unsteered).")

In [ ]:
# [19] Fig 4 — §4 metrics: (a) readout vs obs-change (readable!=controllable), (b) per-step ->target persistence, (c) honest off-manifold resid.
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
COL = {"GT":"k","Unsteered":OK["grey"],"Manifold-global":OK["green"],"PCA geodesic":OK["blue"],"MLP-gradient":OK["pink"]}
names = ["Unsteered","Manifold-global","PCA geodesic","MLP-gradient"]
# (a) readout RMSE vs obs-change toward target: hitting readout != moving obs
ax = axes[0]
for n in names:
    ax.scatter(metrics4[n]["readout"], metrics4[n]["obs_chg"], s=90, color=COL[n], label=n, zorder=3, edgecolor="k")
ax.set_xlabel("readout RMSE (lower = hits position code)"); ax.set_ylabel("obs change vs unsteered")
ax.set_title("(a) readable != controllable"); ax.legend(fontsize=7); style_ax(ax)
# (b) per-step ->target render: persistence / reversion
ax = axes[1]
for n in ["GT","Unsteered","Manifold-global","PCA geodesic","MLP-gradient"]:
    lw = 2.6 if n=="MLP-gradient" else (2.0 if n=="GT" else 1.4)
    ax.plot(steps, to_tgt_step[n], color=COL[n], lw=lw, marker="o", ms=3, label=n)
ax.axvline(4, color=OK["pink"], ls=":", lw=1.2)
ax.text(4.1, ax.get_ylim()[1]*0.9, "MLP-grad\nreverts", color=OK["pink"], fontsize=7)
ax.set_xlabel("rollout step"); ax.set_ylabel("RMS(gen obs, TARGET render)")
ax.set_title("(b) does it reach target & STICK?"); ax.legend(fontsize=7); style_ax(ax)
# (c) honest off-manifold residual per editor vs real reference
ax = axes[2]
ed = ["GT","Manifold-global","PCA geodesic","MLP-gradient"]
vals = [metrics4[n]["honest"] for n in ed]
ax.bar(ed, vals, color=[COL[n] for n in ed], alpha=0.85)
for i,v in enumerate(vals): ax.text(i, v+0.01, f"{v:.2f}", ha="center", fontsize=8)
ax.axhline(real_honest, color="0.3", ls="--", lw=1.4, label=f"real states={real_honest:.2f}")
ax.set_xticklabels(ed, rotation=20, ha="right", fontsize=8)
ax.set_ylabel("honest leave-out local residual"); ax.set_title("(c) on- or off-manifold?"); ax.legend(fontsize=7); style_ax(ax)
fig.suptitle("Fig 4 — Editing head-to-head (GRU): readable!=controllable; MLP-gradient hits obs but off-manifold & reverts", y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig4_editor_metrics.png", dpi=130, bbox_inches="tight"); display(fig); plt.close(fig)

In [ ]:
# [20] Fig 5 — §4 UNIFIED WATERFALL OVERLAY (dark theme): rows=samples, cols=editors. green=target loc, red=ghost.
plt.style.use("dark_background")
def centroid(mask_row):
    idx=np.where(mask_row)[0]; return idx.mean() if idx.size else np.nan
teleport = np.linalg.norm(tgt_pos - pre_pos, axis=-1)[np.arange(N), edit_obj]
has_ghost = ghost_mask.sum(1) >= 3
SAMPLES = list(np.argsort(teleport*has_ghost)[::-1][:3])
order = ["GT","Unsteered","Manifold-global","PCA geodesic","MLP-gradient"]
fig, axes = plt.subplots(len(SAMPLES), len(order), figsize=(3.0*len(order), 3.4*len(SAMPLES)), squeeze=False,
                         facecolor="#0a0a14")
for r,smp in enumerate(SAMPLES):
    tgt_cx = centroid(tgt_render_id[smp]==edit_obj[smp]); pre_cx = centroid(pre_render_id[smp]==edit_obj[smp])
    for c,n in enumerate(order):
        ax = axes[r][c]; ax.set_facecolor("#0a0a14")
        ax.imshow(roll_obs[n][smp], aspect="auto", origin="upper", cmap="magma", vmin=0, vmax=1, interpolation="nearest")
        if not np.isnan(tgt_cx): ax.axvline(tgt_cx, color="#00E676", lw=1.8, alpha=0.95)
        if not np.isnan(pre_cx): ax.axvline(pre_cx, color="#FF5252", ls="--", lw=1.8, alpha=0.95)
        if r==0: ax.set_title(n, fontsize=11, color="w")
        if c==0: ax.set_ylabel(f"smp {smp}\n(teleport {teleport[smp]:.2f})\nrollout frame", fontsize=8, color="w")
        ax.set_xlabel("ray", fontsize=8, color="w"); ax.tick_params(colors="0.7", labelsize=6)
axes[0][0].plot([],[],color="#00E676",lw=2.5,label="target loc (green)")
axes[0][0].plot([],[],color="#FF5252",ls="--",lw=2.5,label="ghost loc (red)")
axes[0][0].legend(loc="upper right", fontsize=7, facecolor="#0a0a14", labelcolor="w")
fig.suptitle("Fig 5 — Unified editor waterfalls: green=where edited obj SHOULD be, red=ghost/original\n"
             "GT = a perfect edit (bright streak at green, none at red). Watch the MLP-gradient streak SNAP BACK to red over frames.",
             y=1.015, fontsize=12, color="w")
fig.tight_layout(); fig.savefig(f"{OUT}/fig5_unified_waterfalls.png", dpi=130, bbox_inches="tight", facecolor="#0a0a14")
display(fig); plt.close(fig); plt.style.use("default")

In [ ]:
# [21] Fig 6 — §4 REVERSION zoom (dark): MLP-gradient waterfall over rollout for the top sample, target/ghost lines.
plt.style.use("dark_background")
smp = SAMPLES[0]
tgt_cx = centroid(tgt_render_id[smp]==edit_obj[smp]); pre_cx = centroid(pre_render_id[smp]==edit_obj[smp])
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), facecolor="#0a0a14")
# (a) GT waterfall (reference), (b) MLP-gradient waterfall (reverts), (c) per-step curves
for ax,(n,ttl) in zip(axes[:2], [("GT","(a) GT (perfect edit: sticks at green)"),
                                 ("MLP-gradient","(b) MLP-gradient (reaches green @step0, snaps to red)")]):
    ax.set_facecolor("#0a0a14")
    ax.imshow(roll_obs[n][smp], aspect="auto", origin="upper", cmap="magma", vmin=0, vmax=1, interpolation="nearest")
    if not np.isnan(tgt_cx): ax.axvline(tgt_cx, color="#00E676", lw=2.0)
    if not np.isnan(pre_cx): ax.axvline(pre_cx, color="#FF5252", ls="--", lw=2.0)
    ax.set_title(ttl, fontsize=10, color="w"); ax.set_xlabel("ray", color="w"); ax.set_ylabel("rollout frame", color="w")
    ax.tick_params(colors="0.7")
ax = axes[2]; ax.set_facecolor("#0a0a14")
ax.plot(steps, to_tgt_step["MLP-gradient"], color="#CC79A7", lw=2.6, marker="o", label="MLP-grad ->target")
ax.plot(steps, to_tgt_step["GT"], color="w", lw=1.6, ls="--", label="GT ->target (floor)")
ax.plot(steps, to_tgt_step["Unsteered"], color="0.5", lw=1.4, label="Unsteered ->target (ceiling)")
ax.axvline(4, color="#CC79A7", ls=":", lw=1.3)
ax.set_xlabel("rollout step", color="w"); ax.set_ylabel("RMS(gen obs, TARGET render)", color="w")
ax.set_title("(c) reversion: reaches target then climbs back", fontsize=10, color="w")
ax.tick_params(colors="0.7"); ax.grid(alpha=0.2); ax.legend(fontsize=8, facecolor="#0a0a14", labelcolor="w")
fig.suptitle("Fig 6 — The reversion example: obs-driven MLP-gradient edit reverts by ~step 4 (off-manifold, does not persist)",
             y=1.03, fontsize=12, color="w")
fig.tight_layout(); fig.savefig(f"{OUT}/fig6_reversion.png", dpi=130, bbox_inches="tight", facecolor="#0a0a14")
display(fig); plt.close(fig); plt.style.use("default")

### §4b — RSSM echo: the same failure, architecture-independent

Two one-shot editors on the **RSSM** (pseudoinverse probe edit + global-manifold edit), same edit set, to
confirm the head-to-head failure is not a GRU idiosyncrasy. Expected (from `rssm_state_geometry`): the
pseudoinverse edit hits the readout **exactly** yet moves the observation ~**0%** (the RSSM position-probe
direction is decoder-inert), and the global-manifold edit moves obs ~36% of a swap but scrambled — i.e.
**readable ≠ controllable is if anything sharper** on the RSSM.

In [ ]:
# [22] §4b — RSSM echo: warm up, probe, pseudoinverse + manifold-global edits, obs-space table.
rmodel = rssm; HR = H_RSSM; states_r = states_rssm
sdefR = StateDefinition(name="positions", state_shape=(N_OBJ,2), extract_fn=lambda b: b["positions"])
linR = LinearExtractor(HR, sdefR, use_lstsq=True); linR.fit(states_r, pos_tf, mask=vis_tf, device=DEVICE)
linR = linR.to(DEVICE).eval(); AR, bR, AR_pinv = probe_decomposition(linR)
subR = fit_state_subspace(states_r, var_threshold=0.90)
subR_dev = replace(subR, mean=subR.mean.to(DEVICE), basis=subR.basis.to(DEVICE),
                   explained_variance_ratio=subR.explained_variance_ratio.to(DEVICE))

warmR = eval.warm_up_to_edit(rmodel, edits.obs[:N], ef, n_viz=N, n_ctx_show=8, device=DEVICE)
h0R = torch.from_numpy(warmR.h_at_edit[:N]).float().to(DEVICE)
def readoutR(h): return h @ AR.T + bR
edit_fnR = lambda h,t: inject_state(h, t, AR, AR_pinv, bR)
h_pinvR = inject_state(h0R, tgt, AR, AR_pinv, bR)
h_maniR = manifold_steer(h0R, tgt, edit_fnR, subR_dev, n_iters=50)

@torch.no_grad()
def rollout_R(h_array, n_rollout):
    obs_all=[]
    for i in range(h_array.shape[0]):
        h = torch.as_tensor(h_array[i], dtype=torch.float32, device=DEVICE).unsqueeze(0)
        o,_ = _rollout(rmodel, h, n_rollout); obs_all.append(o)
    return np.stack(obs_all)
roll_R = {n: rollout_R(h.detach().cpu().numpy(), N_ROLLOUT) for n,h in
          {"Unsteered":h0R, "pseudoinv":h_pinvR, "Manifold-global":h_maniR}.items()}
obs_uR = roll_R["Unsteered"]
gt_swapR = rms(roll_obs["GT"][:,0,:], obs_u[:,0,:])   # reuse GRU GT obs scale for a % reference (structural)
def obs_change_R(obs): return rms(obs[:,0,:], obs_uR[:,0,:])
print("=== §4b RSSM EDITOR ECHO (step 0) ===")
print(f"{'editor':16s} {'readoutRMSE':>11s} {'obs change':>11s}")
for n,h in {"Unsteered":h0R,"pseudoinv":h_pinvR,"Manifold-global":h_maniR}.items():
    rr = float((readoutR(h)-tgt).pow(2).mean().sqrt())
    print(f"{n:16s} {rr:11.4f} {obs_change_R(roll_R[n]):11.4f}")
print("\npseudoinv: readout RMSE ~0 (hits code exactly) yet obs barely moves => readable!=controllable, sharper on RSSM.")

---
## §5 — Synthesis: predictively-sufficient but non-canonical; readable ≠ controllable; architecture-independent

**One-paragraph synthesis.** The learned state has the *dimensionality* of the world (intrinsic ~5–7,
bracketing 8 DOF) but not its *canonicality*: ~34% of `h` is not a function of the minimal `(pos,vel)`
statistic, and the `(pos,vel)→h` embedding is **strongly curved**. Position is nearly-linearly readable and
velocity is nonlinear-**instantaneous** (not temporal). Yet **readable ≠ controllable**: probe/pseudoinverse
edits hit the position *readout* but not the *observation*; the iterative PCA-geodesic stays on-manifold and
moves the obs a lot but does not fully reach target; and the obs-driven MLP-gradient edit hits the target
observation at step 0 only to **revert by ~step 4** (off-manifold, non-persistent). All of this is
**architecture-independent**: a refined KL-regularised RSSM replicates it, with position and the non-canonical
code living in the **deterministic core** (as (non-)canonical as the GRU's `h`) and the stochastic `s` holding
neither. **The KL structure buys no canonicity and no controllability.** This frames — as a *hypothesis* — the
organizing claim that **editability ⟺ a canonical, factored, predictively-sufficient state**, and motivates
explicit physical scaffolding (not stochastic latents) as the route to it.

In [ ]:
# [23] Fig 7 — §5 synthesis dashboard: (a) recoverability vs canonicality, (b) editor readable!=controllable, (c) architecture parity.
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
# (a) recoverable (position/velocity R2) vs canonical (fiber residual) — GRU vs RSSM(det)
ax = axes[0]
groups = ["position\nR2 (MLP)","velocity\nR2 (MLP,late)","canonicality\n1 - fiberResid"]
gru_vals = [pos_res[("GRU","mlp")]["r2"], vel_res[("GRU","late")][("sf","mlp")]["r2"], 1-gru_r]
rssm_vals= [pos_split["RSSM h_det (256)"], vel_res[("RSSM","late")][("sf","mlp")]["r2"], 1-det_r]
x=np.arange(3); w=0.38
ax.bar(x-w/2, gru_vals, w, color=OK["blue"], label="GRU")
ax.bar(x+w/2, rssm_vals, w, color=OK["orange"], label="RSSM (det core)")
ax.set_xticks(x); ax.set_xticklabels(groups, fontsize=8); ax.set_ylim(0,1.02)
ax.set_ylabel("score (higher=better)"); ax.set_title("(a) recoverable & predictive, but only ~65% canonical")
ax.legend(fontsize=8); style_ax(ax)
# (b) editor summary: readout reached vs obs reached (fraction of gap closed)
ax = axes[1]
uns_to_tgt = metrics4["Unsteered"]["to_target"]
def gap_closed(n): return 100*(uns_to_tgt - metrics4[n]["to_target"])/max(uns_to_tgt,1e-9)
def readout_closed(n):
    cold = metrics4["Unsteered"]["readout"]
    return 100*(cold - metrics4[n]["readout"])/max(cold,1e-9)
eds = ["Manifold-global","PCA geodesic","MLP-gradient"]
x=np.arange(len(eds)); w=0.38
ax.bar(x-w/2, [readout_closed(n) for n in eds], w, color=OK["green"], label="readout gap closed %")
ax.bar(x+w/2, [gap_closed(n) for n in eds], w, color=OK["pink"], label="obs->target gap closed %")
ax.set_xticks(x); ax.set_xticklabels(eds, rotation=15, ha="right", fontsize=8); ax.set_ylabel("% gap closed")
ax.axhline(0,color="k",lw=0.8); ax.set_title("(b) readable != controllable (readout>>obs)"); ax.legend(fontsize=7); style_ax(ax)
# (c) architecture parity: fiber residual GRU vs RSSM-det vs RSSM-full vs s
ax = axes[2]
ord2 = ["GRU h","RSSM det","RSSM full","RSSM s"]
vals2 = [gru_r, det_r, full_r, s_r]
cols2 = [OK["blue"],OK["orange"],OK["grey"],OK["yellow"]]
ax.bar(ord2, vals2, color=cols2, alpha=0.85)
for i,v in enumerate(vals2): ax.text(i, v+0.01, f"{v:.2f}", ha="center", fontsize=8)
ax.set_ylabel("fiber residual (lower=more canonical)")
ax.set_title("(c) architecture-independent: det core ~= GRU; KL adds nothing"); style_ax(ax)
fig.suptitle("Fig 7 — Synthesis: predictively-sufficient, non-canonical, readable!=controllable, architecture-independent", y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig7_synthesis.png", dpi=130, bbox_inches="tight"); display(fig); plt.close(fig)

In [ ]:
# [24] §5 — auto master summary table (all headline numbers, both models) + PNG manifest.
print("="*80); print("MASTER EDITABILITY — CONSOLIDATED SUMMARY (GRU + refined RSSM)"); print("="*80)
print("\n[§1 GEOMETRY]  intrinsic dim ~5-7 (TwoNN 5.2 / MLE 6.9) brackets 8 DOF; hull @90%: "
      f"GRU {dims_g[0.90]} / RSSM {dims_r[0.90]}; tangent ~{CITED['tangent_angle_gru']:.0f}deg/{CITED['tangent_angle_rssm']:.0f}deg (curved).")
print("\n[§2 RECOVERABILITY]")
print(f"  position R2:   GRU lin {pos_res[('GRU','lin')]['r2']:.2f}/MLP {pos_res[('GRU','mlp')]['r2']:.2f}   "
      f"RSSM lin {pos_res[('RSSM','lin')]['r2']:.2f}/MLP {pos_res[('RSSM','mlp')]['r2']:.2f}")
for lbl in ["GRU","RSSM"]:
    o=vel_res[(lbl,'late')]
    print(f"  velocity(late) {lbl}: sf-lin {o[('sf','lin')]['r2']:.2f} -> sf-MLP {o[('sf','mlp')]['r2']:.2f}  "
          f"(2f-MLP {o[('win','mlp')]['r2']:.2f}, D={o[('win','mlp')]['r2']-o[('sf','mlp')]['r2']:+.3f}) => nonlinear-instantaneous, NOT temporal")
print("\n[§3 CANONICALITY]  fiber residual (MLP): "
      f"GRU h {gru_r:.3f} ~= RSSM det {det_r:.3f}  (RSSM full {full_r:.3f} inflated by s {s_r:.3f}). ~{gru_r*100:.0f}% non-canonical.")
print("\n[§4 EDITING]  (GRU) readable != controllable:")
print(f"{'editor':16s} {'readoutRMSE':>11s} {'->target':>9s} {'obschg':>8s} {'ghost':>7s} {'honestResid':>12s}")
for n in ["GT","Unsteered","Manifold-global","PCA geodesic","MLP-gradient"]:
    m=metrics4[n]; print(f"{n:16s} {m['readout']:11.3f} {m['to_target']:9.3f} {m['obs_chg']:8.3f} {m['ghost']:7.2f} {m['honest']:12.2f}")
print(f"  MLP-gradient reversion: ->target step0 {to_tgt_step['MLP-gradient'][0]:.3f} -> step4 {to_tgt_step['MLP-gradient'][4]:.3f} (climbs back).")
print(f"  real-state honest off-manifold residual reference = {real_honest:.2f}")
print("\n[§5]  Predictively-sufficient but non-canonical; curved (pos,vel)->h; velocity nonlinear-instantaneous;")
print("      readable != controllable; ARCHITECTURE-INDEPENDENT (RSSM replicates, KL delivers no canonicity/controllability).")
print("\nPNGs saved to /tmp/master_editability/:")
for f in sorted(os.listdir(OUT)):
    if f.endswith(".png"): print("  ", os.path.join(OUT,f))